# 日本大学数据

用日本大学的数据，把 Polars 的基本操作过一遍：读取、列操作、行过滤、分组聚合、画图。与其抽象地讲 API，不如直接在真实数据上完整做一遍。这份数据里有日文列名，有缺失值，也有需要做类型转换的字段，正好覆盖实际工作里最常遇到的几件事。

先读 CSV，显示前两行。把 `polars` 导入成 `pl` 是惯例的写法。

## 读取文件

In [1]:
import polars as pl
import hvplot.polars
df = pl.read_csv('data/japanese_universities.csv')
df.head(2)

,code,name,name_jp,type,type_jp,address,postal_code,phone,state,state_jp,latitude,longitude,found,faculty_count,department_count,has_grad,has_remote,review_rating,review_count,difficulty_SD,difficulty_rank
i64,str,str,str,str,str,str,str,str,str,str,f64,f64,str,i64,i64,bool,bool,f64,f64,f64,str
0,"""F101110100010""","""Hokkaido University""","""北海道大学""","""National""","""国立""","""北海道札幌市北区北8条西5丁目""","""060－0808""","""011－716－2111""","""Hokkai Do""","""北海道""",43.070446,141.347153,"""1876-08""",33,78,true,false,4.16,1389.0,60.4,"""A"""
1,"""F101110100029""","""Hokkaido University of Educati…","""北海道教育大学""","""National""","""国立""","""北海道札幌市北区あいの里5条3－1－3""","""002－8501""","""011－778－0206""","""Hokkai Do""","""北海道""",43.170498,141.393753,"""1943-04""",3,8,true,false,3.79,544.0,47.1,"""D"""


`columns` 返回列名列表，顺序就是文件里的顺序。

## 列的信息

In [2]:
df.columns

['',
 'code',
 'name',
 'name_jp',
 'type',
 'type_jp',
 'address',
 'postal_code',
 'phone',
 'state',
 'state_jp',
 'latitude',
 'longitude',
 'found',
 'faculty_count',
 'department_count',
 'has_grad',
 'has_remote',
 'review_rating',
 'review_count',
 'difficulty_SD',
 'difficulty_rank']

列名和数据类型用 `schema` 确认：它返回一个“列名 → 数据类型”的字典，比只看 `dtypes` 好，因为名字和类型一一对应，不会看错行。

In [3]:
df.schema

Schema([('', Int64),
        ('code', String),
        ('name', String),
        ('name_jp', String),
        ('type', String),
        ('type_jp', String),
        ('address', String),
        ('postal_code', String),
        ('phone', String),
        ('state', String),
        ('state_jp', String),
        ('latitude', Float64),
        ('longitude', Float64),
        ('found', String),
        ('faculty_count', Int64),
        ('department_count', Int64),
        ('has_grad', Boolean),
        ('has_remote', Boolean),
        ('review_rating', Float64),
        ('review_count', Float64),
        ('difficulty_SD', Float64),
        ('difficulty_rank', String)])

## 列操作

`drop()` 用来丢掉不需要的列。Polars 的方法都返回新表，所以要用变量接住结果，原表并不会被改动。

In [4]:
df2 = df.drop('', 'code', 'name', 'type', 'state', 'postal_code', 'phone')
df2.head(2)

name_jp,type_jp,address,state_jp,latitude,longitude,found,faculty_count,department_count,has_grad,has_remote,review_rating,review_count,difficulty_SD,difficulty_rank
str,str,str,str,f64,f64,str,i64,i64,bool,bool,f64,f64,f64,str
"""北海道大学""","""国立""","""北海道札幌市北区北8条西5丁目""","""北海道""",43.070446,141.347153,"""1876-08""",33,78,true,false,4.16,1389.0,60.4,"""A"""
"""北海道教育大学""","""国立""","""北海道札幌市北区あいの里5条3－1－3""","""北海道""",43.170498,141.393753,"""1943-04""",3,8,true,false,3.79,544.0,47.1,"""D"""


`with_columns()` 用来加列，或者转换已有的列，它返回的同样是一张新表。下面的代码对 `df2` 做两种转换，得到 `df3`：

* `found`：从字符串转成日期型，原来的 `YYYY-MM` 文本变成真正能比较大小的日期。
* `type_jp` 和 `difficulty_rank`：从字符串转成类别型。类别型不重复存同样的字符串，内存更省，分组时也不用反复做字符串比较。

传给 `with_columns()` 的不是计算结果，而是“表示计算的表达式”，与 Pandas 的赋值写法差别很大。[表达式的细节](expression)

In [5]:
df3 = df2.with_columns(
    pl.col('found').str.to_date("%Y-%m"),
    pl.col('type_jp', 'difficulty_rank').cast(pl.Categorical)
)

用 `select()` 只取出转换过的列，确认结果是否正确。

In [6]:
df3.select('name_jp', 'found', 'type_jp', 'difficulty_rank').head()

name_jp,found,type_jp,difficulty_rank
str,date,cat,cat
"""北海道大学""",1876-08-01,"""国立""","""A"""
"""北海道教育大学""",1943-04-01,"""国立""","""D"""
"""室蘭工業大学""",1897-05-01,"""国立""","""F"""
"""小樽商科大学""",1910-03-01,"""国立""","""C"""
"""帯広畜産大学""",1941-04-01,"""国立""","""B"""


:::{tip}
`select()` 和 `with_columns()` 都是做列操作的方法，区别只在输出：`select()` 只保留选中的列，其余列全部丢弃，而 `with_columns()` 把新算出的列并进原表，没被操作的列原样保留。要“换一张表”用前者，要“在表上追加内容”用后者。
:::

## 行过滤

`filter()` 只返回满足条件的行，原表不变。下面的代码筛选偏差值(`difficulty_SD`)大于 65 的行，也就是偏差值较高的一批学校。

In [7]:
df3.filter(
    pl.col('difficulty_SD') > 65
)

name_jp,type_jp,address,state_jp,latitude,longitude,found,faculty_count,department_count,has_grad,has_remote,review_rating,review_count,difficulty_SD,difficulty_rank
str,cat,str,str,f64,f64,date,i64,i64,bool,bool,f64,f64,f64,cat
"""東京大学""","""国立""","""東京都文京区本郷7－3－1""","""東京都""",35.714146,139.763214,1877-04-01,25,132,true,false,4.34,2206.0,70.5,"""S"""
"""一橋大学""","""国立""","""東京都国立市中2－1""","""東京都""",35.694389,139.443649,1920-04-01,10,15,true,false,4.26,432.0,67.9,"""S"""
"""京都大学""","""国立""","""京都府京都市左京区吉田本町""","""京都府""",35.026962,135.781967,1886-04-01,28,94,true,false,4.2,1434.0,65.6,"""S"""
"""国際教養大学""","""公立""","""秋田県秋田市雄和椿川字奥椿岱""","""秋田県""",39.62706,140.181107,2004-04-01,2,4,true,false,4.18,100.0,69.7,"""S"""
"""慶應義塾大学""","""私立""","""東京都港区三田2－15－45""","""東京都""",35.649147,139.742828,1890-01-01,28,59,true,true,4.19,2682.0,66.3,"""S"""
"""日本医科大学""","""私立""","""東京都文京区千駄木1－1－5""","""東京都""",35.721233,139.758453,1904-04-01,2,2,true,false,3.99,31.0,70.0,"""S"""
"""早稲田大学""","""私立""","""東京都新宿区戸塚町１－１０４""","""東京都""",35.710083,139.722244,1902-10-01,35,94,true,true,4.14,4280.0,65.7,"""S"""
"""国際基督教大学""","""私立""","""東京都三鷹市大沢3－10－2""","""東京都""",35.686474,139.527481,1953-03-01,2,6,true,false,4.43,206.0,68.0,"""S"""


多个条件用 `&`(AND)、`|`(OR)、`~`(NOT)组合。注意每个条件都要用括号包起来，否则运算符优先级会让表达式算错。下面的代码筛出位于大阪、且偏差值高于 55 的大学。

In [8]:
df3.filter(
    (pl.col('difficulty_SD') > 55) & (pl.col('state_jp').str.starts_with("大阪"))
)

name_jp,type_jp,address,state_jp,latitude,longitude,found,faculty_count,department_count,has_grad,has_remote,review_rating,review_count,difficulty_SD,difficulty_rank
str,cat,str,str,f64,f64,date,i64,i64,bool,bool,f64,f64,f64,cat
"""大阪大学""","""国立""","""大阪府吹田市山田丘1－1""","""大阪府""",34.819496,135.521164,1917-04-01,26,75,true,false,4.12,1840.0,61.3,"""A"""
"""大阪医科薬科大学""","""私立""","""大阪府高槻市大学町2－7""","""大阪府""",34.85186,135.624405,1904-05-01,6,9,true,false,4.11,9.0,57.0,"""B"""
"""関西大学""","""私立""","""大阪府吹田市山手町3－3－35""","""大阪府""",34.769669,135.510162,1904-01-01,29,44,true,false,3.95,2661.0,55.6,"""B"""


方法链就是把多个方法连着写。Polars 从上到下的顺序就是处理顺序，中间结果一直往下传，所以写预处理时这是基本形态，不需要给每一步都起中间变量名。

下面的代码对 `df3` 依次做：

* `filter()`：只留下 `type_jp` 为“国立”的行。
* `sort()`：按偏差值降序。
* `drop_nulls()`：去掉偏差值为 NULL 的行。
* `select()`：只留大学名和偏差值。
* `head()`：取前 5 行。

In [9]:
(df3
.filter(pl.col.type_jp == '国立')
.sort('difficulty_SD', descending=True)
.drop_nulls('difficulty_SD')
.select('name_jp', 'difficulty_SD')
.head(5)
)

name_jp,difficulty_SD
str,f64
"""東京大学""",70.5
"""一橋大学""",67.9
"""京都大学""",65.6
"""東京工業大学""",65.0
"""浜松医科大学""",64.4


## 分组处理

先看 `type_jp` 有哪些取值。结果只有国立、公立、私立三种，说明这一列正好适合当分组键。

In [10]:
df3.select(pl.col('type_jp').unique())

type_jp
cat
"""国立"""
"""私立"""
"""公立"""


按分类列聚合时用 `group_by()` 和 `agg()`：前者负责“怎么分”，后者负责“每组算什么”。

* `group_by()`：按指定列的值把数据分组。
* `agg()`：对每组做聚合(计数、求和、平均等)。

下面的例子用 `pl.len()` 数每组的行数。

In [11]:
df3.group_by('type_jp').agg(pl.len())

type_jp,len
cat,u32
"""私立""",626
"""国立""",86
"""公立""",101


不只计数，偏差值的最小、中位、平均、最大可以一次算出来，比反复调用不同的 `agg` 再拼表要简单得多。用关键字参数指定输出列名即可。

In [12]:
(df3
.group_by('type_jp')
.agg(
    count = pl.len(),
    min = pl.col('difficulty_SD').min(),
    median = pl.col('difficulty_SD').median(),
    mean = pl.col('difficulty_SD').mean(),
    max = pl.col('difficulty_SD').max(),
)
)

type_jp,count,min,median,mean,max
cat,u32,f64,f64,f64,f64
"""私立""",626,35.0,38.5,40.801034,70.0
"""公立""",101,40.0,50.0,50.042391,69.7
"""国立""",86,35.7,51.7,52.917073,70.5


## 画图

Polars 本身不画图，但装上 `hvplot` 之后可以直接从 `.hvplot` 出交互图，不必先转成 Pandas。要叠地图还需要 `geoviews`，因为它提供地理坐标和投影的支持。

```
conda install -c conda-forge hvplot geoviews
```

### 柱状图

下面的代码按都道府县数大学的个数，从多到少排序，画成柱状图。画图方法都挂在 `.hvplot` 下，返回的是一个可交互的图对象，而不是一张静态图片。

* 用 `x` 和 `y` 指定轴标签。
* `rot=90` 让 X 轴标签竖排，避免都道府县名互相重叠。
* `frame_width=900` 把图宽设成 900 像素。

In [13]:
x_label = "都道府县"
y_label = "大学数量"
(df3
.group_by(pl.col("state_jp").alias(x_label))
.agg(
    pl.len().alias(y_label)
)
.sort(y_label, descending=True)
.hvplot.bar(
    x=x_label, y=y_label,
    rot=90, frame_width=900
)
)

:Bars   [都道府县]   (大学数量)

### 堆叠柱状图

只看总数还不够：同样多的大学，可能集中在国立，也可能全是私立。这次按大学种类(国立、公立、私立)把柱子堆起来，和上面的代码有三点不同。

1. `group_by()` 里加上 `type_jp`，按“都道府县 × 种类”分组。
2. 排序按“每个都道府县的总数”来，而不是按单根柱子的高度，用 `over()` 在表达式里做分组聚合就能做到([over() 的用法](over))。
3. `by="type_jp"` 按种类分色，`stacked=True` 堆叠显示。

In [14]:
(df3
.group_by(
    pl.col("state_jp").alias(x_label), 
    "type_jp"
)
.agg(
    pl.len().alias(y_label)
)
.sort(pl.col(y_label).sum().over(x_label), descending=True)
.hvplot.bar(
    x=x_label, y=y_label, by="type_jp", 
    stacked=True, rot=90, 
    frame_width=900, frame_height=450
)
)

:Bars   [都道府县,type_jp]   (大学数量)

最后把经纬度打成点，用大学种类分色，叠到地图上。

* `by="type_jp"`：按种类分色。
* `hover_cols=['name_jp']`：鼠标悬停时显示大学名。
* `tiles=True`：铺上地图底图。

注意不要同时写 `geo=True`：在 geoviews 1.15 / pyproj 3.8 这套组合下，它会让投影计算直接报错，只写 `tiles=True` 就能出地图。

In [15]:
df3.hvplot.points(
    "longitude",
    "latitude",
    by="type_jp",
    hover_cols=["name_jp"],
    tiles=True,
    frame_width=600,
)

:Overlay
   .Tiles.I     :Tiles   [x,y]
   .NdOverlay.I :NdOverlay   [type_jp]
      :Points   [x,y]   (name_jp)